In [1]:
import time
import os
import mysql.connector
import numpy as np
import csv
from scipy.stats import ttest_rel,t

In [2]:
#connect to mySQL
conn =mysql.connector.connect(
    host="localhost",
    user="root",
    password=os.environ.get("MYSQL_PASSWORD"),
    database="mydb"
)

In [3]:
cursor=conn.cursor()

In [4]:
# my list of(query_name,baseline_query,optimized_query)
queries=[
    (
        "i want my most frequent customers",
        """
        SELECT c.customer_id,c.name,COUNT(o.order_id) AS total_orders
        FROM customer c
        JOIN `order` o ON c.customer_id=o.customer_id
        GROUP BY c.customer_id
        ORDER BY total_orders DESC;
        """,
        """
        EXPLAIN FORMAT=JSON
        SELECT c.customer_id,c.name,o.total_orders
        FROM customer c
        JOIN (
            SELECT customer_id,COUNT(*) AS total_orders
            FROM `order`
            GROUP BY customer_id
        ) o ON c.customer_id=o.customer_id
        ORDER BY o.total_orders DESC;
        """
    ),
    (
        "most popular products/most sold products",
        """
        SELECT p.product_id,p.name,SUM(oi.quantity) AS total_sold
        FROM product p
        JOIN orderitem oi ON p.product_id=oi.product_id
        GROUP BY p.product_id
        ORDER BY total_sold DESC;
        """,
        """
        WITH total_sales AS (
            SELECT product_id,SUM(quantity) AS total_sold
            FROM orderitem
            GROUP BY product_id
        )
        SELECT p.product_id,p.name,t.total_sold
        FROM product p
        JOIN total_sales t ON p.product_id=t.product_id
        ORDER BY t.total_sold DESC;
        """
    ),
    (
        "highest spending customers",
        """
        SELECT c.customer_id,c.name,SUM(p.amount) AS total_spent
        FROM customer c
        JOIN `order` o ON c.customer_id=o.customer_id
        JOIN payment p ON o.order_id=p.order_id
        GROUP BY c.customer_id
        ORDER BY total_spent DESC;
        """,
        """
        SELECT c.customer_id,c.name,SUM(po.total_spent) AS total_spent
        FROM customer c
        JOIN `order` o ON c.customer_id=o.customer_id
        JOIN (
            SELECT order_id,SUM(amount) AS total_spent
            FROM payment
            GROUP BY order_id
        ) po ON o.order_id=po.order_id
        GROUP BY c.customer_id
        ORDER BY total_spent DESC;
        """
    ),
    (
        "customers placed the most orders in the last 6 months",
        """
        SELECT c.customer_id,c.name,COUNT(o.order_id) as total_order
        FROM Customer c
        JOIN `Order` o ON c.customer_id=o.customer_id
        WHERE o.order_date>='2024-12-11'
        GROUP BY(customer_id)
        order by(total_order) DESC;
        """,
        """
        SELECT c.customer_id,c.name,recent_orders.total_order
        FROM Customer c
        JOIN (
            SELECT customer_id,COUNT(order_id) AS total_order
            FROM `Order`
            WHERE order_date>=CURDATE()-INTERVAL 6 MONTH
            GROUP BY customer_id
    ) recent_orders ON c.customer_id=recent_orders.customer_id
        ORDER BY recent_orders.total_order DESC;
        """
    ),
    (
        "most frequently bought product pair",
        """
        SELECT oi1.product_id AS product_1,oi2.product_id AS product_2,COUNT(*) AS bought_together
        FROM Orderitem oi1
        JOIN Orderitem oi2 ON oi1.order_id=oi2.order_id AND oi1.product_id<oi2.product_id
        GROUP BY oi1.product_id,oi2.product_id
        ORDER BY bought_together DESC;
        """,
        """
        SELECT pairs.product_1,pairs.product_2,COUNT(*) AS bought_together
        FROM (
            SELECT oi1.order_id,oi1.product_id AS product_1,oi2.product_id AS product_2
            FROM OrderItem oi1
            JOIN OrderItem oi2 
              ON oi1.order_id=oi2.order_id 
             AND oi1.product_id<oi2.product_id
        ) AS pairs
        GROUP BY pairs.product_1,pairs.product_2
        ORDER BY bought_together DESC;
        """
    ),
    (
        "total sales per category",
        """
        SELECT cat.category_id,cat.category_name,SUM(oi.quantity*oi.price) AS total_sales
        FROM category cat
        JOIN productcategory pc ON cat.category_id=pc.category_id
        JOIN product p ON pc.product_id=p.product_id
        JOIN orderitem oi ON p.product_id=oi.product_id
        GROUP BY cat.category_id,cat.category_name;
        """,
        """
        WITH product_revenue AS (
          SELECT product_id,SUM(quantity*price) AS revenue
          FROM orderitem
          GROUP BY product_id
        )
        SELECT cat.category_id,cat.category_name,SUM(pr.revenue) AS total_sales
        FROM product_revenue pr
        JOIN productcategory pc ON pr.product_id=pc.product_id
        JOIN category cat ON pc.category_id=cat.category_id
        GROUP BY cat.category_id,cat.category_name;
        """
    ),
    (
        "total payment per customer",
        """
        SELECT c.customer_id,c.name,SUM(p.amount) as total_payment 
        FROM Customer c 
        JOIN `Order` o ON c.customer_id = o.customer_id 
        JOIN Payment p ON o.order_id = p.order_id 
        GROUP BY c.customer_id;
        """,
        """
        SELECT payments.customer_id,c.name,payments.total_payment 
        FROM (
            SELECT o.customer_id,SUM(p.amount) AS total_payment 
            FROM `Order` o 
            JOIN Payment p ON o.order_id=p.order_id 
            GROUP BY o.customer_id
        ) AS payments 
        JOIN Customer c ON payments.customer_id=c.customer_id;
        """
    )
]

In [5]:
results=[]
for name,baseline_query,optimized_query in queries:
    baseline_runtimes=[]
    optimized_runtimes=[]
    for _ in range(20):
        start=time.time()
        cursor.execute(baseline_query)
        cursor.fetchall()
        baseline_runtimes.append(time.time()-start)
    for _ in range(20):
        start=time.time()
        cursor.execute(optimized_query)
        cursor.fetchall()
        optimized_runtimes.append(time.time()-start)
        
    #converting to np arrays
    b=np.array(baseline_runtimes)
    o=np.array(optimized_runtimes)
    #paired t-test
    t_stat,p_value=ttest_rel(b, o)
    #confidence interval(95%) for mean diff
    diff=b-o
    mean_diff=np.mean(diff)
    std_err=np.std(diff,ddof=1)/np.sqrt(len(diff))
    ci_margin=t.ppf(0.975,df=len(diff)-1)*std_err
    ci_lower=mean_diff-ci_margin
    ci_upper=mean_diff+ci_margin

    results.append([
        name,round(mean_diff,4),round(ci_lower,4),round(ci_upper,4),
        round(p_value,6)
    ])


In [6]:
#saving to CSV
with open("after_indexing_project2_stats_results.csv","w",newline="") as f:
    writer=csv.writer(f)
    writer.writerow([
        "Query Name","Mean Diff (B - O)",
        "95% CI Lower","95% CI Upper","p-value(t-test)"
    ])
    writer.writerows(results)
print("after indexing statistical analysis complete. Results saved to after_indexing_project2_stats_results.csv")


after indexing statistical analysis complete. Results saved to after_indexing_project2_stats_results.csv
